In [2]:
# Scrape the website API

import requests
import pandas as pd
import time

all_levels = []
page = 1

while True:
    try:
        response = requests.get(
            "https://api.tuforums.com/v2/database/levels",
            params={"page": page, "limit": 500}, timeout=10
        )
    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")
        break
    data = response.json()
    all_levels.extend(data["results"])

    print(f"Page {page} done | Total levels fetched so far: {len(all_levels)}")

    if not data["hasMore"]:
        break
    page += 1
    time.sleep(0.1)  # be polite to their server

print(f"Pulled {len(all_levels)} levels total")

Page 1 done | Total levels fetched so far: 100
Page 2 done | Total levels fetched so far: 200
Page 3 done | Total levels fetched so far: 300
Page 4 done | Total levels fetched so far: 400
Page 5 done | Total levels fetched so far: 500
Page 6 done | Total levels fetched so far: 600
Page 7 done | Total levels fetched so far: 700
Page 8 done | Total levels fetched so far: 800
Page 9 done | Total levels fetched so far: 900
Page 10 done | Total levels fetched so far: 1000
Page 11 done | Total levels fetched so far: 1100
Page 12 done | Total levels fetched so far: 1200
Page 13 done | Total levels fetched so far: 1300
Page 14 done | Total levels fetched so far: 1400
Page 15 done | Total levels fetched so far: 1500
Page 16 done | Total levels fetched so far: 1600
Page 17 done | Total levels fetched so far: 1700
Page 18 done | Total levels fetched so far: 1800
Page 19 done | Total levels fetched so far: 1900
Page 20 done | Total levels fetched so far: 2000
Page 21 done | Total levels fetched so

In [3]:
TIER_VALUES = {"P": 0, "G": 20, "U": 40} # Assign values for each tier so that the AI may be fed data easily. 

rated_levels = []


for level in all_levels:
    diff = level.get("difficulty")
    level_id = level["id"]
    level["dlLink"] = f"https://api.tuforums.com/v2/database/levels/{level_id}/level.adofai"
    if isinstance(diff, dict):
        tier = diff.get("name")
        tier_upper = tier.upper() if tier else "" # Removing Q Tiers
        # Skip junk tiers
        if tier and not tier_upper.startswith(("Q", "PQ", "GQ", "UQ")) and tier not in ["SPECIAL", "Censored", "Impossible", "Unranked"]:
            level["difficulty"] = tier
            level["tuforums_link"] = f"https://tuforums.com/levels/{level.get('id')}" # Link to TUForums ADOFAI chart

            ms = level.get("levelLengthInMs")
            if ms is None or float(ms) == 0:
                print(f"Skipping level {level_id}: missing/invalid levelLengthInMs")
                continue
            level["levelLengthInMs"] = round(float(ms) / 1000, 1) # Calculates song duration

            tile_count = level.get("tilecount")
            if tile_count is None or float(tile_count) == 0:
                print(f"Skipping level {level_id}: missing/invalid tilecount")
                continue
            tile_count_val = float(tile_count)

            level["density"] = ms / tile_count_val # Chart density formula

            # Convert PGU ratings into numbers for AI training (NOT LEGACY ADOFAI RATING) e.g. P5 = 5, G10 = 30 (20+10), U10 = 50 (40+10)
            tier_letter = tier_upper[0]
            tier_number = tier[1:]

            if tier_letter in TIER_VALUES and tier_number.isdigit():
                level["difficulty_number"] = TIER_VALUES[tier_letter] + int(tier_number)
            else:
                print(f"Could not parse difficulty '{tier}' for level {level_id}, skipping")
                continue  

            rated_levels.append(level)



Skipping level 16217: missing/invalid levelLengthInMs
Skipping level 15823: missing/invalid levelLengthInMs
Skipping level 15741: missing/invalid levelLengthInMs
Skipping level 14966: missing/invalid levelLengthInMs
Skipping level 14268: missing/invalid levelLengthInMs
Skipping level 14092: missing/invalid levelLengthInMs
Skipping level 14089: missing/invalid levelLengthInMs
Skipping level 14085: missing/invalid levelLengthInMs
Skipping level 14084: missing/invalid levelLengthInMs
Skipping level 14068: missing/invalid levelLengthInMs
Skipping level 13905: missing/invalid levelLengthInMs
Skipping level 13864: missing/invalid levelLengthInMs
Skipping level 13851: missing/invalid levelLengthInMs
Skipping level 13472: missing/invalid levelLengthInMs
Skipping level 13409: missing/invalid levelLengthInMs
Skipping level 10531: missing/invalid levelLengthInMs
Skipping level 10149: missing/invalid levelLengthInMs
Skipping level 9968: missing/invalid levelLengthInMs
Skipping level 9963: missing/

In [7]:
df = pd.DataFrame(rated_levels)
print(df.shape)

(3779, 56)


In [ ]:
# Function for parsing .adofai levels (twirl count, speed change count)

def extract_gameplay_features(level_json):
    actions = level_json.get("actions", [])
    
    twirl_count = 0
    speed_change_count = 0
    
    for action in actions:
        event_type = action.get("eventType")
        if event_type == "Twirl":
            twirl_count += 1
        elif event_type == "SetSpeed":
            speed_change_count += 1
    
    return {
        "twirl_count": twirl_count,
        "speed_change_count": speed_change_count,
    }

In [27]:
# Parsing .adofai levels

import time

gameplay_features = []

for i, row in df.iterrows():
    try:
        resp = requests.get(row["dlLink"], timeout=10)
        level_json = resp.json()

        # Filters any errors
        if "error" in level_json:
            print(f"Skipping id {row['id']}: {level_json['error']}")
            continue

        features = extract_gameplay_features(level_json)
        features["id"] = row["id"]
        gameplay_features.append(features)
    except Exception as e:
        print(f"Error: Failed on id {row['id']} ({e})")

    time.sleep(0.5)
print(f"Succesfully gained features for {len(gameplay_features)} levels.")

Skipping id 16253: Level .adofai file not found
Skipping id 15603: Level .adofai file not found
Skipping id 15358: Level .adofai file not found
Skipping id 15199: Level .adofai file not found
Skipping id 14911: Level .adofai file not found
Skipping id 14134: Level .adofai file not found
Skipping id 13917: Level .adofai file not found
Skipping id 13706: Level .adofai file not found
Skipping id 13481: Level .adofai file not found
Skipping id 13259: Level .adofai file not found
Skipping id 13246: Level .adofai file not found
Skipping id 13245: Level .adofai file not found
Skipping id 13240: Level .adofai file not found
Skipping id 13211: Level .adofai file not found
Skipping id 13209: Level .adofai file not found
Skipping id 13208: Level .adofai file not found
Skipping id 13188: Level .adofai file not found
Skipping id 13186: Level .adofai file not found
Skipping id 13184: Level .adofai file not found
Skipping id 13182: Level .adofai file not found
Skipping id 13181: Level .adofai file no

In [29]:
gameplay_df = pd.DataFrame(gameplay_features)
print(gameplay_df.shape)

df = df.merge(gameplay_df, on="id", how="left")
print(df.shape)

print(df[["twirl_count", "speed_change_count"]].isna().sum())

(3738, 3)
(3779, 58)
twirl_count           41
speed_change_count    41
dtype: int64


In [24]:
print(df[df["id"] == 16253]["dlLink"].values[0])

https://api.tuforums.com/v2/database/levels/16253/level.adofai


In [ ]:
import sqlite3

DB_PATH = "levels.db"

def get_db():
    con = sqlite3.connect(DB_PATH, timeout=10)
    con.row_factory = sqlite3.Row
    return con

In [37]:
df.dropna(subset=["twirl_count", "speed_change_count"]) # Drops missing values in twirl_count and speed_change_count

schema_columns = [
    "id", "song", "creator", "difficulty", "difficulty_number", "tilecount",
    "levelLengthInMs", "bpm", "tuforums_link", "dlLink", "density", "twirl_count", "speed_change_count"
]


df_to_save = df.reindex(columns=schema_columns)
print(df_to_save.shape)

with get_db() as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS raw_levels (
            id INTEGER PRIMARY KEY,
            song TEXT,
            creator TEXT,
            difficulty REAL,
            difficulty_number REAL,
            tilecount INTEGER,
            levelLengthInMs INTEGER,
            bpm REAL,
            tuforums_link TEXT,
            dlLink TEXT,
            density INTEGER,
            twirl_count INTEGER,
            speed_change_count INTEGER
        )    
    """)


    df_to_save.to_sql("raw_levels", con, if_exists="replace", index=False)
print("Databased saved for deep learning.")


(3738, 13)
Databased saved for deep learning.


In [38]:
import os
print(os.path.getsize("levels.db") / 1024, "KB")

4460.0 KB
